# 16 — Production NLP Patterns
**Goal:** Empaquetar, serializar y desplegar un pipeline NLP de producción con patrones robustos.

```
Custom Transformers    → componentes reutilizables compatibles con sklearn
Pipeline serialization → guardar y cargar modelos entrenados (pickle / joblib)
Serving patterns       → API de predicción, batch vs online
Monitoring             → detectar drift, logging de predicciones
Testing                → unit tests para componentes NLP
```

In [ ]:
import os
import re
import time
import joblib
import pickle
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import deque

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('nlp_pipeline')

MODELS_DIR = Path('models/nlp')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup completo.')

## 1. Custom Transformers — componentes reutilizables

In [ ]:
STOPWORDS_ES = frozenset([
    'a','al','como','con','de','del','el','en','es','la','las','lo','los',
    'me','mi','muy','no','o','para','por','que','se','si','sin','su','te',
    'todo','un','una','y','yo',
])


class SpanishTextCleaner(BaseEstimator, TransformerMixin):
    """Limpieza de texto en español — stateless, siempre seguro de encadenar."""

    def __init__(self, remove_urls=True, remove_emojis=True,
                 lowercase=True, min_token_len=2):
        self.remove_urls    = remove_urls
        self.remove_emojis  = remove_emojis
        self.lowercase      = lowercase
        self.min_token_len  = min_token_len

    def fit(self, X, y=None):
        return self   # nada que aprender

    def _clean(self, text):
        if self.lowercase:
            text = text.lower()
        if self.remove_urls:
            text = re.sub(r'https?://\S+', '', text)
        if self.remove_emojis:
            text = re.sub(r'[^\x00-\x7Fáéíóúüñ\s]', '', text)
        text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
        tokens = [t for t in text.split() if len(t) >= self.min_token_len]
        return ' '.join(tokens)

    def transform(self, X):
        return [self._clean(text) for text in X]


class TextLengthFilter(BaseEstimator, TransformerMixin):
    """Marca textos demasiado cortos para tratamiento especial."""

    def __init__(self, min_words=3):
        self.min_words = min_words

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # En lugar de filtrar (rompería el alineamiento con y),
        # agrega un marcador especial
        result = []
        for text in X:
            if len(text.split()) < self.min_words:
                result.append(f'__SHORT__ {text}')
            else:
                result.append(text)
        return result


# Probar
cleaner = SpanishTextCleaner()
samples = [
    'LLEVO 3 DÍAS SIN RESPUESTA!!! 😤 https://queja.com',
    'ok',
    'La app está MUY lenta, tardé 20 min en activar la tarjeta',
]
for original, cleaned in zip(samples, cleaner.transform(samples)):
    print(f'  Original: {original}')
    print(f'  Cleaned:  {cleaned}\n')

## 2. Pipeline completo — entrenar y serializar

In [ ]:
# Dataset
templates_pos = [
    'Excelente servicio, aprobaron mi tarjeta rápidamente',
    'La app funciona perfecto, muy fácil de usar',
    'Proceso de activación muy sencillo y sin problemas',
    'Increíbles beneficios, recomiendo la tarjeta',
    'Atención al cliente excepcional, resolvieron todo',
]
templates_neg = [
    'La app está muy lenta, tardé mucho en activar la tarjeta',
    'No me llegó el OTP, tuve que llamar varias veces',
    'La carga de documentos falla constantemente',
    'El call center no responde, pésimo servicio',
    'Llevo días esperando la evaluación crediticia sin respuesta',
]

rng = np.random.default_rng(42)
texts  = [templates_pos[rng.integers(5)] for _ in range(300)] + \
         [templates_neg[rng.integers(5)] for _ in range(300)]
labels = [1]*300 + [0]*300

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Pipeline de producción con todos los componentes
sentiment_pipeline = Pipeline([
    ('cleaner', SpanishTextCleaner()),
    ('length_filter', TextLengthFilter(min_words=3)),
    ('tfidf', TfidfVectorizer(
        stop_words=list(STOPWORDS_ES),
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
        max_features=10_000,
    )),
    ('clf', LogisticRegression(max_iter=1000, C=1.0)),
], verbose=False)

sentiment_pipeline.fit(X_train, y_train)
y_pred = sentiment_pipeline.predict(X_test)

print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=['Negativo','Positivo']))

In [ ]:
# Serializar con joblib (más eficiente que pickle para objetos numpy)
model_path = MODELS_DIR / 'sentiment_v1.joblib'
joblib.dump(sentiment_pipeline, model_path)
print(f'Modelo guardado: {model_path} ({model_path.stat().st_size/1024:.1f} KB)')

# Cargar y verificar
loaded_pipeline = joblib.load(model_path)
y_pred_loaded = loaded_pipeline.predict(X_test[:5])
print(f'Predicciones del modelo cargado: {y_pred_loaded}')

## 3. Model versioning — metadata con el modelo

In [ ]:
@dataclass
class ModelArtifact:
    """Empaqueta pipeline + metadatos para versionado."""
    pipeline:    object
    version:     str
    task:        str
    labels:      List[str]
    train_f1:    float
    val_f1:      float
    vocab_size:  int
    trained_at:  str = field(default_factory=lambda: pd.Timestamp.now().isoformat())
    description: str = ''

    def predict(self, texts):
        return self.pipeline.predict(texts)

    def predict_proba(self, texts):
        return self.pipeline.predict_proba(texts)

    def save(self, path):
        joblib.dump(self, path)
        print(f'Artifact saved: {path}')

    @classmethod
    def load(cls, path):
        return joblib.load(path)

    def info(self):
        print(f'Model: {self.task} v{self.version}')
        print(f'  Labels:    {self.labels}')
        print(f'  Val F1:    {self.val_f1:.4f}')
        print(f'  Vocab:     {self.vocab_size:,}')
        print(f'  Trained:   {self.trained_at}')

# Empaquetar
artifact = ModelArtifact(
    pipeline   = sentiment_pipeline,
    version    = '1.0.0',
    task       = 'sentiment',
    labels     = ['negativo', 'positivo'],
    train_f1   = f1_score(y_train, sentiment_pipeline.predict(X_train)),
    val_f1     = f1_score(y_test, y_pred),
    vocab_size = len(sentiment_pipeline.named_steps['tfidf'].vocabulary_),
    description = 'Sentimiento en comentarios NPS — fintech tarjeta de crédito',
)

artifact.info()
artifact.save(MODELS_DIR / 'sentiment_v1.0.0.joblib')

## 4. Serving — predicción online y batch

In [ ]:
class NLPPredictor:
    """Wrapper de predicción con logging, validación y fallback."""

    def __init__(self, artifact: ModelArtifact, fallback_label=None):
        self.artifact       = artifact
        self.fallback_label = fallback_label
        self._call_log: List[Dict] = []

    def predict_one(self, text: str) -> Dict:
        """Predicción online — un texto a la vez."""
        t0 = time.perf_counter()

        # Validación de entrada
        if not text or not text.strip():
            return {'label': self.fallback_label, 'confidence': 0.0,
                    'error': 'empty_input', 'latency_ms': 0}

        try:
            proba = self.artifact.predict_proba([text])[0]
            pred  = int(np.argmax(proba))
            conf  = float(proba[pred])
            label = self.artifact.labels[pred]
        except Exception as e:
            logger.error(f'Prediction error: {e}')
            return {'label': self.fallback_label, 'confidence': 0.0, 'error': str(e)}

        latency = (time.perf_counter() - t0) * 1000
        result  = {'label': label, 'confidence': conf,
                   'probas': dict(zip(self.artifact.labels, proba.round(4).tolist())),
                   'latency_ms': round(latency, 2)}
        self._call_log.append({'text_len': len(text), **result})
        return result

    def predict_batch(self, texts: List[str], batch_size=64) -> List[Dict]:
        """Predicción batch — lista de textos."""
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            results.extend([self.predict_one(t) for t in batch])
        return results

    def latency_stats(self):
        latencies = [r['latency_ms'] for r in self._call_log if 'error' not in r]
        if not latencies:
            return
        print(f'Latencia ({len(latencies)} predicciones):')
        print(f'  p50: {np.percentile(latencies, 50):.2f} ms')
        print(f'  p95: {np.percentile(latencies, 95):.2f} ms')
        print(f'  p99: {np.percentile(latencies, 99):.2f} ms')

predictor = NLPPredictor(artifact, fallback_label='desconocido')

# Online
test_inputs = [
    'La tarjeta llegó rapidísimo, excelente servicio!',
    'Llevan días sin resolver mi problema',
    '',                                         # entrada vacía
    'ok',                                       # muy corta
]

print('Predicciones online:\n')
for text in test_inputs:
    result = predictor.predict_one(text)
    print(f'  Input: "{text}"')
    print(f'  → {result}\n')

In [ ]:
# Batch prediction con latency benchmark
batch_texts = X_test[:100]
t0 = time.perf_counter()
batch_results = predictor.predict_batch(batch_texts)
total_ms = (time.perf_counter() - t0) * 1000

print(f'Batch de {len(batch_texts)} textos: {total_ms:.1f} ms total ({total_ms/len(batch_texts):.2f} ms/texto)')
predictor.latency_stats()

## 5. Data drift detection — monitorear el modelo en producción

In [ ]:
class PredictionMonitor:
    """Detecta drift en la distribución de predicciones."""

    def __init__(self, window_size=100, alert_threshold=0.2):
        self.window       = deque(maxlen=window_size)
        self.baseline_pos = None   # proporción positiva en train
        self.threshold    = alert_threshold

    def set_baseline(self, predictions):
        self.baseline_pos = np.mean(predictions)
        print(f'Baseline % positivos: {self.baseline_pos:.2%}')

    def log(self, predictions):
        self.window.extend(predictions)

    def check_drift(self):
        if self.baseline_pos is None or len(self.window) == 0:
            return
        current_pos = np.mean(list(self.window))
        drift = abs(current_pos - self.baseline_pos)
        status = '⚠️  DRIFT' if drift > self.threshold else '✅  OK'
        print(f'{status} | Baseline: {self.baseline_pos:.2%} | Actual: {current_pos:.2%} | Δ: {drift:+.2%}')
        return drift

monitor = PredictionMonitor(window_size=100, alert_threshold=0.15)
train_preds = sentiment_pipeline.predict(X_train)
monitor.set_baseline(train_preds)

# Simular datos normales
normal_preds = sentiment_pipeline.predict(X_test)
monitor.log(normal_preds)
monitor.check_drift()

# Simular drift — de repente llegan solo comentarios negativos
drift_preds = [0] * 80  # mayoría negativos
monitor.log(drift_preds)
monitor.check_drift()

## 6. Unit tests para componentes NLP

In [ ]:
def test_cleaner():
    c = SpanishTextCleaner()
    assert c.transform(['HOLA'])[0] == 'hola'
    assert 'http' not in c.transform(['visita https://foo.com'])[0]
    # texto vacío
    assert c.transform([''])[0] == ''
    # preservar ñ y tildes
    result = c.transform(['niño español'])[0]
    assert 'niño' in result
    print('test_cleaner: PASS')

def test_pipeline_shapes():
    p = sentiment_pipeline
    preds = p.predict(X_test)
    assert len(preds) == len(X_test)
    assert set(preds).issubset({0, 1})
    probas = p.predict_proba(X_test)
    assert probas.shape == (len(X_test), 2)
    assert np.allclose(probas.sum(axis=1), 1.0, atol=1e-6)
    print('test_pipeline_shapes: PASS')

def test_predictor_fallback():
    p = NLPPredictor(artifact, fallback_label='desconocido')
    r = p.predict_one('')
    assert r['label'] == 'desconocido'
    assert 'error' in r
    print('test_predictor_fallback: PASS')

def test_serialization():
    path = MODELS_DIR / '_test_artifact.joblib'
    artifact.save(path)
    loaded = ModelArtifact.load(path)
    preds_orig   = artifact.predict(X_test[:10])
    preds_loaded = loaded.predict(X_test[:10])
    assert (preds_orig == preds_loaded).all()
    path.unlink()   # limpiar
    print('test_serialization: PASS')

# Correr todos
for test_fn in [test_cleaner, test_pipeline_shapes, test_predictor_fallback, test_serialization]:
    try:
        test_fn()
    except AssertionError as e:
        print(f'{test_fn.__name__}: FAIL — {e}')

## 7. Checklist de producción

In [ ]:
checklist = {
    'Pipeline versionado con metadata':          True,
    'Serialización joblib (no pickle puro)':     True,
    'Validación de entrada (texto vacío/corto)': True,
    'Fallback ante errores de predicción':       True,
    'Logging de predicciones':                   True,
    'Latency tracking (p50/p95/p99)':            True,
    'Drift detection en distribución':           True,
    'Unit tests de componentes':                 True,
    'max_features para limitar RAM':             True,
    'Baseline de métricas documentado':          True,
}

print('Checklist de producción NLP:')
for item, done in checklist.items():
    mark = '✅' if done else '❌'
    print(f'  {mark}  {item}')

## Resumen
| Patrón | Implementación |
|---|---|
| Custom transformer | `BaseEstimator + TransformerMixin` — compatible con Pipeline |
| Serialización | `joblib.dump/load` — más eficiente que pickle para numpy |
| Model artifact | Dataclass con pipeline + metadatos + versión |
| Online serving | `predict_one()` con validación y fallback |
| Batch serving | Loop con batches para controlar memoria |
| Drift detection | Comparar distribución actual vs baseline con ventana deslizante |
| Testing | `assert` básicos sobre shapes, labels, serialización |

---
## Curso completo — índice de notebooks

| # | Tema | Nivel |
|---|---|---|
| 01 | Text Basics | Fundamentos |
| 02 | Bag of Words | Fundamentos |
| 03 | TF-IDF | Fundamentos |
| 04 | Similarity | Core |
| 05 | Sentiment | Core |
| 06 | Topic Modeling | Core |
| 07 | Text Classification | Core |
| 08 | Real World Pipeline | Core |
| 09 | Text Normalization | Avanzado |
| 10 | Clustering | Avanzado |
| 11 | LSA / Embeddings | Avanzado |
| 12 | Language Models | Avanzado |
| 13 | Information Retrieval / BM25 | Avanzado |
| 14 | Feature Engineering | Avanzado |
| 15 | Attention desde cero | Maestría |
| 16 | Production Patterns | Maestría |